# Inspect DICOM images — local & Azure Blob

Lightweight, read-only viewer for **data inspection** of CT (and other) DICOM files, sourced
either from a **local path** or an **Azure Blob** container. It applies the correct pixel
transforms (RescaleSlope/Intercept → HU, VOI windowing) and previews single instances,
full series, and de-identification-relevant header tags.

> ⚠️ **PHI awareness.** Real DICOM headers carry PHI (patient name/ID, dates, institution) and
> images can contain burned-in annotations. Inspect **de-identified** data by default; treat any
> identified data per `docs/03-phi-detection-deidentification.md`. This notebook never writes
> pixels back and only prints a curated, non-identifying subset of tags unless you opt in.

## Environment (uv-managed, aligned kernel)

From `prototype/model-training/ct-report-pretraining/`:

```bash
uv sync --extra dicom          # pydicom + matplotlib + azure-storage-blob + azure-identity + jupyter
uv run python -m ipykernel install --user \
  --name ct-report-pretraining --display-name "CT-Report (uv)"
```

Then pick **Kernel → CT-Report (uv)**.

## 0. Imports & display helpers

`window()` applies `RescaleSlope`/`RescaleIntercept` (→ Hounsfield Units for CT) and a
center/width VOI window. Presets cover common CT views; pass explicit values for other
modalities. `SAFE_TAGS` is the non-identifying subset printed by default.

In [ ]:
from __future__ import annotations
import io, os, glob, tempfile
import numpy as np
import matplotlib.pyplot as plt
import pydicom
from pydicom.pixel_data_handlers.util import apply_modality_lut, apply_voi_lut

# Common CT window presets: (center, width) in HU.
CT_WINDOWS = {
    "soft_tissue": (40, 400),
    "lung": (-600, 1500),
    "bone": (300, 1500),
    "brain": (40, 80),
    "mediastinum": (50, 350),
}

# Non-identifying header tags safe to print during inspection.
SAFE_TAGS = [
    "Modality", "SOPClassUID", "Rows", "Columns", "PixelSpacing", "SliceThickness",
    "RescaleSlope", "RescaleIntercept", "WindowCenter", "WindowWidth", "PhotometricInterpretation",
    "BitsStored", "InstanceNumber", "SeriesNumber", "BodyPartExamined", "ImageOrientationPatient",
]

def to_hu(ds: pydicom.Dataset) -> np.ndarray:
    """Decode pixels and apply the modality LUT (Rescale slope/intercept → HU for CT)."""
    arr = ds.pixel_array.astype(np.float32)
    return apply_modality_lut(arr, ds)

def window(ds: pydicom.Dataset, preset: str | None = "soft_tissue",
           center: float | None = None, width: float | None = None) -> np.ndarray:
    """Return a 0..1 windowed image. Explicit center/width override the preset."""
    hu = to_hu(ds)
    if center is None or width is None:
        if preset and preset in CT_WINDOWS:
            center, width = CT_WINDOWS[preset]
        else:
            # fall back to the file's stored VOI window (or full range).
            try:
                return _norm(apply_voi_lut(ds.pixel_array.astype(np.float32), ds))
            except Exception:
                center, width = float(np.mean(hu)), float(np.ptp(hu) or 1.0)
    lo, hi = center - width / 2.0, center + width / 2.0
    return np.clip((hu - lo) / (hi - lo), 0.0, 1.0)

def _norm(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.float32); mn, mx = float(a.min()), float(a.max())
    return (a - mn) / (mx - mn) if mx > mn else np.zeros_like(a)

def show(ds: pydicom.Dataset, preset="soft_tissue", center=None, width=None, title=None, ax=None):
    img = window(ds, preset, center, width)
    ax = ax or plt.subplots(figsize=(5, 5))[1]
    ax.imshow(img, cmap="gray"); ax.axis("off")
    ax.set_title(title or f"{getattr(ds,'Modality','?')}  {preset or f'{center}/{width}'}", fontsize=9)
    return ax

def print_tags(ds: pydicom.Dataset, tags=SAFE_TAGS):
    for t in tags:
        if t in ds:
            print(f"  {t:26s}: {ds.get(t)}")

print("helpers ready — windows:", list(CT_WINDOWS))

## 1. Local: view a single DICOM file

Point `LOCAL_FILE` at a `.dcm` on disk. Prints the safe header subset and renders the default
soft-tissue window.

In [ ]:
LOCAL_FILE = "/path/to/instance.dcm"   # <- set me

ds = pydicom.dcmread(LOCAL_FILE)
print(f"{os.path.basename(LOCAL_FILE)}  shape={ds.pixel_array.shape}  dtype={ds.pixel_array.dtype}")
print_tags(ds)
show(ds, preset="soft_tissue", title=os.path.basename(LOCAL_FILE))
plt.show()

### 1a. Compare windows on the same slice

Handy to confirm HU calibration and pick the right VOI for the finding you're inspecting.

In [ ]:
presets = ["soft_tissue", "lung", "bone", "brain"]
fig, axes = plt.subplots(1, len(presets), figsize=(4 * len(presets), 4))
for ax, p in zip(np.atleast_1d(axes), presets):
    show(ds, preset=p, title=p, ax=ax)
plt.tight_layout(); plt.show()

## 2. Local: view a series (folder of slices)

Reads every DICOM in `LOCAL_SERIES_DIR`, sorts by `ImageOrientationPatient`+`ImagePositionPatient`
(falling back to `InstanceNumber`), and renders an axial montage. Reuse the loaded stack for
quick QC (spacing, intensity range, slice count).

In [ ]:
LOCAL_SERIES_DIR = "/path/to/series_dir"   # <- set me

def load_series(paths):
    slices = [pydicom.dcmread(p) for p in paths]
    slices = [s for s in slices if hasattr(s, "PixelData")]
    def key(s):
        ipp = getattr(s, "ImagePositionPatient", None)
        if ipp is not None:
            return float(ipp[2])
        return float(getattr(s, "InstanceNumber", 0))
    slices.sort(key=key)
    return slices

def montage(slices, preset="soft_tissue", cols=6, max_slices=24):
    n = min(len(slices), max_slices)
    idx = np.linspace(0, len(slices) - 1, n).astype(int)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.2 * rows))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, i in zip(np.atleast_1d(axes).ravel(), idx):
        show(slices[i], preset=preset, title=f"z={i}", ax=ax)
    plt.tight_layout(); plt.show()

paths = sorted(glob.glob(os.path.join(LOCAL_SERIES_DIR, "**", "*.dcm"), recursive=True))
series = load_series(paths)
print(f"loaded {len(series)} slices from {LOCAL_SERIES_DIR}")
if series:
    s0 = series[0]
    print(f"  spacing={s0.get('PixelSpacing')}  thickness={s0.get('SliceThickness')}")
    montage(series, preset="lung")

## 3. Azure Blob: credential-less inspection

Reads DICOM straight from a blob container using **`DefaultAzureCredential`** (your `az login`
or a managed identity) — no account keys or SAS, matching the identity-based access in
`azureml/datastore.yml` and `infra/modules/storage.bicep` (`allowSharedKeyAccess=false`).

**Prereq:** your identity needs **Storage Blob Data Reader** on the account/container.
Set the account + container below; the Gold zone container is `ct-report-gold`.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

STORAGE_ACCOUNT = "REPLACE_STORAGE_ACCOUNT"   # <- set me (no .blob.core.windows.net)
CONTAINER = "ct-report-gold"
PREFIX = ""   # optional virtual-folder prefix to scope the listing

cred = DefaultAzureCredential()
svc = BlobServiceClient(f"https://{STORAGE_ACCOUNT}.blob.core.windows.net", credential=cred)
container = svc.get_container_client(CONTAINER)

def list_dicom(prefix=PREFIX, limit=50):
    hits = []
    for b in container.list_blobs(name_starts_with=prefix):
        if b.name.lower().endswith(".dcm") or "dicom" in b.name.lower():
            hits.append(b.name)
            if len(hits) >= limit:
                break
    return hits

blobs = list_dicom()
print(f"found {len(blobs)} DICOM blob(s) under '{CONTAINER}/{PREFIX}'")
for n in blobs[:10]:
    print("  ", n)

### 3a. Read one blob into memory and view

`pydicom.dcmread` accepts a file-like object, so we stream bytes with no temp file on disk.

In [ ]:
def read_blob_dataset(name: str) -> pydicom.Dataset:
    data = container.get_blob_client(name).download_blob().readall()
    return pydicom.dcmread(io.BytesIO(data))

if blobs:
    ds_blob = read_blob_dataset(blobs[0])
    print(blobs[0])
    print_tags(ds_blob)
    show(ds_blob, preset="soft_tissue", title=os.path.basename(blobs[0]))
    plt.show()
else:
    print("no DICOM blobs found — adjust CONTAINER / PREFIX above.")

### 3b. View a blob-hosted series

Downloads the blobs under a prefix into memory, sorts, and renders the montage — the same
`load_series`/`montage` helpers, fed from Blob instead of local disk.

In [ ]:
SERIES_PREFIX = PREFIX   # <- narrow to one study/series folder for a coherent stack

names = list_dicom(prefix=SERIES_PREFIX, limit=200)
datasets = [read_blob_dataset(n) for n in names]
datasets = [d for d in datasets if hasattr(d, "PixelData")]
def zkey(s):
    ipp = getattr(s, "ImagePositionPatient", None)
    return float(ipp[2]) if ipp is not None else float(getattr(s, "InstanceNumber", 0))
datasets.sort(key=zkey)
print(f"downloaded {len(datasets)} slice(s) from blob")
if datasets:
    montage(datasets, preset="lung")

## 4. De-identification spot-check (opt-in)

Scans a dataset for common **PHI-bearing** tags so you can confirm data is de-identified before
wider use. Prints tag values — run only on data you're authorized to view. Aligns with the
DICOM PS3.15 profile discussion in `docs/03` / `docs/12`.

In [ ]:
PHI_TAGS = [
    "PatientName", "PatientID", "PatientBirthDate", "PatientAddress", "PatientTelephoneNumbers",
    "OtherPatientIDs", "AccessionNumber", "StudyDate", "SeriesDate", "AcquisitionDate",
    "InstitutionName", "InstitutionAddress", "ReferringPhysicianName", "PerformingPhysicianName",
    "OperatorsName", "StationName", "DeviceSerialNumber",
]

def phi_scan(ds: pydicom.Dataset):
    present = [(t, ds.get(t)) for t in PHI_TAGS if t in ds and str(ds.get(t)).strip()]
    if not present:
        print("✅ no populated PHI tags from the checklist — looks de-identified (tags only; not a pixel scan).")
    else:
        print(f"⚠️ {len(present)} populated PHI tag(s) — treat as identified data:")
        for t, v in present:
            print(f"  {t:26s}: {v}")
    print("  (Reminder: burned-in pixel annotations are NOT detected here.)")

# phi_scan(ds)        # local dataset from section 1
# phi_scan(ds_blob)   # blob dataset from section 3a